Import the libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

Load the data

In [2]:
df = pd.read_csv("online_shoppers_intention.csv")
print(df.shape)

(12330, 18)


Remove duplicate rows

In [3]:
df = df.drop_duplicates()
print(df.shape)

(12205, 18)


Convert True/False to 1/0

In [4]:
df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)
print(df[["Weekend", "Revenue"]].head())

   Weekend  Revenue
0        0        0
1        0        0
2        0        0
3        0        0
4        1        0


Separate features (X) and target (y)

In [5]:
X = df.drop(columns=["Revenue"])
y = df["Revenue"]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (12205, 17)
y shape: (12205,)


Split into training data (80%) and test data (20%)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=27,
    stratify=y
)
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (9764, 17)
Test: (2441, 17)


Check the buyer percentage in both sets

In [7]:
print("Train buyers %:", round(y_train.mean() * 100, 2))
print("Test buyers %:", round(y_test.mean() * 100, 2))

Train buyers %: 15.63
Test buyers %: 15.65


Save the test data as test_data.csv

In [8]:
test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv("test_data.csv", index=False)
print("Saved test_data.csv with shape:", test_df.shape)

Saved test_data.csv with shape: (2441, 18)


Import the tools

In [9]:
import os
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report)

List the column groups

In [10]:
categorical_cols = ["Month", "VisitorType", "OperatingSystems",
                    "Browser", "Region", "TrafficType"]

numeric_cols = ["Administrative", "Administrative_Duration",
                "Informational", "Informational_Duration",
                "ProductRelated", "ProductRelated_Duration",
                "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

print(len(categorical_cols), "categorical,", len(numeric_cols), "numeric")

6 categorical, 10 numeric


Build the preprocessor

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    remainder="passthrough"
)

A reusable function for the 6 metrics

In [12]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "AUC":       roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "MCC":       matthews_corrcoef(y_test, y_pred),
    }
    return results, y_pred

Model 1: Train Logistic Regression

In [13]:
lr_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

lr_model.fit(X_train, y_train)
print("Logistic Regression trained!")

Logistic Regression trained!


Evaluate it

In [14]:
lr_results, lr_pred = evaluate_model(lr_model, X_test, y_test)

for metric, value in lr_results.items():
    print(f"{metric}: {value:.4f}")

Accuracy: 0.8812
AUC: 0.8954
Precision: 0.7347
Recall: 0.3770
F1: 0.4983
MCC: 0.4702


Confusion matrix and classification report

In [15]:
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

[[2007   52]
 [ 238  144]]
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      2059
           1       0.73      0.38      0.50       382

    accuracy                           0.88      2441
   macro avg       0.81      0.68      0.72      2441
weighted avg       0.87      0.88      0.86      2441



Save the model

In [16]:
os.makedirs("model", exist_ok=True)
joblib.dump(lr_model, "model/logistic_regression.pkl")
print("Saved!")

Saved!


Train the other 4 models. 

Import the 4 models

In [17]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.base import clone

Model 2: Decision Tree

In [20]:
dt_model = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(max_depth=6, random_state=27))
])
dt_model.fit(X_train, y_train)

dt_results, dt_pred = evaluate_model(dt_model, X_test, y_test)
for metric, value in dt_results.items():
    print(f"{metric}: {value:.4f}")
print(confusion_matrix(y_test, dt_pred))

joblib.dump(dt_model, "model/decision_tree.pkl")

Accuracy: 0.9009
AUC: 0.9185
Precision: 0.7096
Recall: 0.6204
F1: 0.6620
MCC: 0.6061
[[1962   97]
 [ 145  237]]


['model/decision_tree.pkl']

Model 3: K-Nearest Neighbors

In [21]:
knn_model = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])
knn_model.fit(X_train, y_train)

knn_results, knn_pred = evaluate_model(knn_model, X_test, y_test)
for metric, value in knn_results.items():
    print(f"{metric}: {value:.4f}")
print(confusion_matrix(y_test, knn_pred))

joblib.dump(knn_model, "model/knn.pkl")

Accuracy: 0.8705
AUC: 0.8063
Precision: 0.6486
Recall: 0.3770
F1: 0.4768
MCC: 0.4285
[[1981   78]
 [ 238  144]]


['model/knn.pkl']

Model 4: Naive Bayes

In [22]:
nb_model = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("classifier", GaussianNB())
])
nb_model.fit(X_train, y_train)

nb_results, nb_pred = evaluate_model(nb_model, X_test, y_test)
for metric, value in nb_results.items():
    print(f"{metric}: {value:.4f}")
print(confusion_matrix(y_test, nb_pred))

joblib.dump(nb_model, "model/naive_bayes.pkl")

Accuracy: 0.2536
AUC: 0.5990
Precision: 0.1709
Recall: 0.9791
F1: 0.2911
MCC: 0.1169
[[ 245 1814]
 [   8  374]]


['model/naive_bayes.pkl']

Model 5: Support Vector Machine

In [23]:
svm_model = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("classifier", SVC(probability=True, random_state=27))
])
svm_model.fit(X_train, y_train)

svm_results, svm_pred = evaluate_model(svn_model, X_test, y_test)
for metric, value in svm_results.items():
    print(f"{metric}: {value:.4f}")
print(confusion_matrix(y_test, svm_pred))

joblib.dump(svm_model, "model/svm.pkl")

e:\BITS_Pilani\Jyotisha\Machine Learning\VS_Code\ml-assignment-2\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


NameError: name 'svn_model' is not defined